# ParetoBandit Demo Playground

**ParetoBandit** is an adaptive LLM router that uses a *contextual bandit* algorithm
(Disjoint LinUCB) to learn, in real time, which language model gives the best
quality-per-dollar for each incoming prompt.

### The core idea

You have access to several LLMs — say, a cheap model (Llama-8B), a mid-tier model
(Mistral-Large), and a premium model (Gemini-Pro). Each request has a *context*
(the embedded prompt). The router learns a mapping from context to expected quality
for every model, and picks the one that maximises a utility score that balances
**quality** against **cost**.

### Three knobs you can turn

| Parameter | What it controls | Low value | High value |
|-----------|-----------------|-----------|------------|
| `alpha` | **Exploration vs. exploitation.** Controls the width of the Upper Confidence Bound (UCB). | Exploits current best estimates — lower variance, risk of getting stuck. | Explores more aggressively — higher variance, discovers better options faster. |
| `forgetting_factor` | **Adaptation speed.** A geometric discount (0 < γ ≤ 1) applied to older observations. | Forgets faster — adapts quickly to distribution shifts, but noisier. | Forgets slower (1.0 = never forget) — more stable, but slow to react to changes. |
| `cost_penalty` | **Cost aversion.** A static weight λ_c that penalises expensive models in the UCB score. | Quality-focused — picks the best model regardless of cost. | Cost-focused — strongly prefers cheaper models. |

### The BudgetPacer

On top of the static `cost_penalty`, the **BudgetPacer** provides an *adaptive*
Lagrangian multiplier that dynamically adjusts the cost penalty on every request
to hit a target average spend ($/request). If spending is above target, it
ratchets up the penalty to steer traffic toward cheaper models; if below, it
relaxes it to unlock higher-quality (more expensive) routes.

### What this notebook covers

1. Load evaluation data (shipped with the library)
2. Run your first routing trial and see model selection fractions
3. Experiment with budget-paced routing
4. Sweep parameters and observe the quality-cost trade-off
5. Run the full pre-built scenarios
6. Bring your own data

---
## 1. Setup

This notebook requires the `[demo]` extra, which includes the embedding model
(`sentence-transformers`) and `matplotlib`:

```bash
pip install paretobandit[demo]
```

If you cloned the repo and installed in editable mode (`pip install -e ".[demo]"`),
you are already set.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from pareto_bandit.demo import (
    DemoConfig,
    DataSplit,
    load_evaluation_data,
    run_trial,
    run_scenario_1,
    run_scenario_2,
    ARM_ORDER,
    ARM_SHORT,
    ARM_COLORS,
)
from pareto_bandit.feature_service import FeatureService
from pareto_bandit.budget_pacer import BudgetPacer, PacingMode

print("Imports OK")

---
## 2. Load and Inspect the Evaluation Data

ParetoBandit ships a **test holdout** of 1,824 prompts drawn from public
benchmarks (GSM8K, WinoGrande, MMLU, etc.). Each prompt has been evaluated
against three LLM "arms":

| Arm | Typical cost | Role |
|-----|-------------|------|
| **Llama-8B** | ~\$0.00003 / req | Budget — fast, cheap, good on easy tasks |
| **Mistral-Large** | ~\$0.0005 / req | Mid-tier — strong generalist |
| **Gemini-Pro** | ~\$0.015 / req | Premium — best on hard reasoning, but 500× more expensive than Llama |

- **Reward** (0–1): how correct / high-quality the model's response was.
- **Cost** (USD): the actual API cost for that request.

The data is split 2:1 into train (online learning) and test (evaluation).

> **Try it:** Change `n_prompts` to use fewer (faster) or more (smoother) prompts.
> Point `prompts_file` at your own JSONL to use custom data.

In [ ]:
fs = FeatureService()

train, test = load_evaluation_data(
    prompts_file=DemoConfig().prompts_file,
    feature_service=fs,
    n_prompts=1000,
    seed=42,
)

print(f"Train: {train.n} prompts   Test: {test.n} prompts")
print(f"Features per prompt: {train.embeddings.shape[1] - 1} + 1 bias\n")

print(f"{'Model':<18s}  {'Avg Reward':>10s}  {'Avg Cost (USD)':>14s}")
print("-" * 46)
for arm in ARM_ORDER:
    r = np.mean(train.rewards[arm])
    c = np.mean(train.costs[arm])
    print(f"{ARM_SHORT[arm]:<18s}  {r:>10.3f}  ${c:>13.6f}")

---
## 3. Your First Routing Trial

A **trial** works in two phases:

1. **Online learning** — the router sees each training prompt, picks a model,
   observes the reward and cost, and updates its internal LinUCB estimates.
2. **Evaluation** — the router continues routing on held-out prompts (still
   learning — standard bandit protocol). We measure average reward, cost,
   and which models were selected.

### What the parameters do

- **`alpha`** (default 0.01): the LinUCB exploration coefficient. Higher values
  widen the confidence bound, making the router try under-explored models more
  often. Lower values make it stick with its current best estimate.

- **`forgetting_factor`** (default 0.997): each time a new observation arrives,
  all past observations are discounted by this factor. At 0.997, the effective
  window is ~333 observations. Set to 1.0 for a stationary (never-forget)
  bandit, or 0.99 for aggressive adaptation.

- **`cost_penalty`** (default 0.3): a static weight subtracted from each arm's
  UCB score proportional to its predicted cost. At 0.0 the router ignores cost
  entirely (quality-only); at 1.0 it strongly prefers cheap models.

Below we run three trials to show the effect of `cost_penalty`:

In [ ]:
for cp_label, cp_value in [("quality-only", 0.0), ("balanced", 0.3), ("cost-focused", 1.0)]:
    trial = run_trial(
        train, test,
        alpha=0.01,
        forgetting_factor=0.997,
        cost_penalty=cp_value,
        seed=42,
    )
    fracs = ", ".join(
        f"{ARM_SHORT[a]}={trial.model_fractions[a]:.0%}" for a in ARM_ORDER
    )
    print(
        f"cost_penalty={cp_value:.1f} ({cp_label:>13s}):  "
        f"reward={trial.mean_reward:.4f}  "
        f"cost=${trial.mean_cost:.6f}  "
        f"[{fracs}]"
    )

Notice how:
- At `cost_penalty=0.0`, the router picks the highest-quality model regardless of price.
- At `cost_penalty=1.0`, it routes almost everything to the cheapest model (Llama-8B).
- The default (0.3) strikes a balance.

> **Try it:** Change `alpha` to 0.001 (greedy) or 0.1 (exploratory) and re-run.
> Or set `forgetting_factor=1.0` to disable forgetting.

---
## 4. Budget-Paced Routing

The `cost_penalty` knob above is static — you set it once and it never changes.
In production, what you usually care about is a **budget target**: "I want to
spend at most \$X per request on average."

The **BudgetPacer** solves this. It maintains an adaptive Lagrangian multiplier
λ_s that adjusts *every request*:

- If cumulative spending is **above** the target → λ_s increases → expensive
  models get penalised more → traffic shifts to cheaper models.
- If cumulative spending is **below** the target → λ_s decreases → the router
  is free to pick higher-quality (more expensive) models.

This gives you **budget compliance** without sacrificing more quality than
necessary. Below, we sweep five budget targets from the cheapest model's mean
cost up to the most expensive, and plot the resulting quality-cost frontier.

> **Try it:** Edit `budget_targets` to add tighter or looser budgets and see
> how the router adapts.

In [ ]:
budget_targets = np.geomspace(3e-5, 1.5e-2, num=5)

sweep_rewards, sweep_costs = [], []

for target in budget_targets:
    pacer = BudgetPacer(
        target_avg_spend_usd=target,
        mode=PacingMode.ADAPTIVE,
    )
    trial = run_trial(
        train, test,
        cost_penalty=0.0,
        budget_pacer=pacer,
        seed=42,
    )
    sweep_rewards.append(trial.mean_reward)
    sweep_costs.append(trial.mean_cost)
    print(f"  target=${target:.2e}  ->  reward={trial.mean_reward:.4f}  cost=${trial.mean_cost:.2e}")

# Plot the Pareto frontier
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(sweep_costs, sweep_rewards, "o-", color="#0072B2", linewidth=2, markersize=8)
for t, r, c in zip(budget_targets, sweep_rewards, sweep_costs):
    ax.annotate(f"${t:.1e}", (c, r), textcoords="offset points",
                xytext=(8, -4), fontsize=8, color="0.4")
ax.set_xlabel("Avg Cost per Request (USD)", fontsize=11)
ax.set_ylabel("Mean Reward (Quality)", fontsize=11)
ax.set_xscale("log")
ax.set_title("Budget-Paced Quality vs. Cost Frontier", fontweight="bold")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 5. "What If" Parameter Sweep

Let's systematically sweep one parameter at a time to see how it shapes the
model mix. Below we sweep `cost_penalty` and show the resulting model selection
fractions alongside reward and cost.

**What to expect:**
- As `cost_penalty` increases, the fraction of **Llama-8B** (cheapest) rises
  and **Gemini-Pro** (most expensive) drops.
- Reward decreases because cheaper models are generally lower quality.
- Cost drops sharply.

> **Try it:** Replace `"cost_penalty"` with `"alpha"` or `"forgetting_factor"`
> in the code below and adjust the values list to explore those dimensions.

In [ ]:
sweep_param = "cost_penalty"
sweep_values = [0.0, 0.1, 0.3, 0.5, 1.0]
defaults = dict(alpha=0.01, forgetting_factor=0.997, cost_penalty=0.3)

print(f"{'Value':>8s}  {'Reward':>7s}  {'Cost (USD)':>11s}  ", end="")
for a in ARM_ORDER:
    print(f"{ARM_SHORT[a]:>14s}", end="")
print()
print("-" * 80)

for val in sweep_values:
    kwargs = {**defaults, sweep_param: val}
    trial = run_trial(train, test, seed=42, **kwargs)
    print(f"{val:>8.3f}  {trial.mean_reward:>7.4f}  ${trial.mean_cost:>10.6f}  ", end="")
    for a in ARM_ORDER:
        print(f"{trial.model_fractions[a]:>13.1%}", end=" ")
    print()

---
## 6. Run a Full Built-In Scenario

The demo ships four pre-built scenarios that produce publication-quality
multi-panel plots:

| # | Scenario | What it demonstrates |
|---|----------|---------------------|
| 1 | **Budget-Paced Routing** | Quality-cost Pareto frontier, budget compliance, model allocation across 7 budget levels. |
| 2 | **Quality Degradation & Recovery** | Mistral-Large quality drops in Phase 2 — the router detects the regression via geometric forgetting and shifts traffic; Phase 3 recovers. |
| 3 | **Cost Drift & Recovery** | Gemini-Pro price drops 50× in Phase 2 — the BudgetPacer exploits cheap premium routing; Phase 3 restores normal pricing. |
| 4 | **Configuration Comparison** | Side-by-side effect of alpha, forgetting_factor, and cost_penalty on the model selection mix. |

Each scenario runs multiple independent seeds for smooth curves. You can
customise the run by creating a `DemoConfig` with the values you want.

> **Tip:** From the command line, you can run all scenarios at once:
> ```bash
> paretobandit-demo                    # all 4 scenarios
> paretobandit-demo --scenario 2       # just one
> ```

In [ ]:
from IPython.display import Image, display

cfg = DemoConfig(
    n_prompts=1000,
    n_seeds=3,
    output_dir="demo_results",
)

out_path = run_scenario_1(cfg, train, test)
display(Image(filename=str(out_path)))

> **Try it:** Replace `run_scenario_1` with `run_scenario_2`, `run_scenario_3`,
> or `run_scenario_4` to explore the other scenarios. Adjust `cfg.alpha`,
> `cfg.forgetting_factor`, or `cfg.n_seeds` to see how results change.

---
## 7. Bring Your Own Data

To use your own evaluation data, prepare a **JSONL** file where each line is a
JSON object with this structure:

```json
{
  "prompt": "What is 2 + 2?",
  "arms": {
    "meta-llama/llama-3.1-8b-instruct": {"reward": 1.0, "cost": 0.00002},
    "mistralai/mistral-large-2512":     {"reward": 1.0, "cost": 0.00030},
    "google/gemini-2.5-pro":            {"reward": 1.0, "cost": 0.01000}
  }
}
```

Each record must include all three arm IDs with a `reward` (0–1) and `cost`
(USD) per arm.

### Using a custom encoder

By default, prompts are embedded with `all-MiniLM-L6-v2` + PCA to 25 dims.
To use a different SentenceTransformer model, create a custom `FeatureService`:

```python
from sentence_transformers import SentenceTransformer

st_model = SentenceTransformer("all-mpnet-base-v2")
custom_fs = FeatureService(
    custom_encoder=lambda text: st_model.encode(
        text, normalize_embeddings=True, show_progress_bar=False
    ),
    embedding_dim=st_model.get_sentence_embedding_dimension(),
)
```

Then pass it to `load_evaluation_data()`.

In [ ]:
# Uncomment and edit to use your own data:

# my_train, my_test = load_evaluation_data(
#     prompts_file="path/to/my_rewards.jsonl",
#     feature_service=fs,        # or custom_fs from above
#     n_prompts=1000,
#     seed=42,
# )
#
# cfg = DemoConfig(output_dir="my_results", n_seeds=3)
# out = run_scenario_1(cfg, my_train, my_test)
# display(Image(filename=str(out)))

print("Edit this cell to load your own JSONL data and re-run!")